# Rotation Invariance Sweep

Full 3-D Euler angle sweep to diagnose rotation invariance failures.

For each pre-rotation **R(α, β, γ)** applied to the source, the model should predict
approximately **R⁻¹** to align it to the reference.  
The **residual angle** = angle(R_pred · R_applied) measures how far off the prediction is.

| Section | Question |
|---|---|
| 2 — Single real scan sweep | At which euler angles does the model fail? |
| 3 — Visualisations | Heatmap slices, 1-D marginals, interactive 3-D scatter |
| 4 — Validation sample sweep | Same analysis on a synthetic val pair (known GT) |
| 5 — Multi-sample comparison | Does failure mode generalise across scans / val pairs? |

**Resolution knob** (cell below): default `STEP_DEG = 30` → 12³ = 1 728 runs.  
For a quick first look use `STEP_DEG = 45` (8³ = 512); for fine detail use `STEP_DEG = 10` (36³ = 46 656 — very slow).

In [ ]:
import os, sys, itertools, pickle

EXP_DIR  = os.path.dirname(os.path.abspath('__file__'))
ROOT_DIR = os.path.dirname(os.path.dirname(EXP_DIR))
sys.path.insert(0, EXP_DIR)
sys.path.insert(0, ROOT_DIR)
os.chdir(EXP_DIR)

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import plotly.graph_objects as go
from scipy.spatial.transform import Rotation
from tqdm.auto import tqdm

from config_dowsampled import make_cfg
from dataset import train_valid_data_loader
from model import create_model

from geotransformer.utils.data import precompute_data_stack_mode
from geotransformer.modules.ops import point_to_node_partition, index_select
from geotransformer.modules.ops.transformation import apply_transform

DEVICE = 'cuda'

cfg = make_cfg()
cfg.data.dataset_root = os.path.join(ROOT_DIR, 'data', 'faces')

_, val_loader, neighbor_limits = train_valid_data_loader(cfg, distributed=False)
print('neighbor_limits:', neighbor_limits)

CKPT = os.path.join(
    ROOT_DIR, 'output',
    'geotransformer.facesdownsampledfixed.stage4.gse.k3.max.oacl.stage2.sinkhorn.vl',
    'snapshots', 'epoch-35.pth.tar',
)
model = create_model(cfg).to(DEVICE)
model.neighbor_limits = neighbor_limits
ckpt = torch.load(CKPT, map_location=DEVICE)
sd = ckpt.get('model', ckpt)
sd = {k.replace('module.', ''): v for k, v in sd.items()}
model.load_state_dict(sd, strict=False)
model.eval()
print(f'Loaded: {os.path.basename(CKPT)}')

with torch.no_grad():
    mean_ref_pts = model.generate_reference_geometry(torch.zeros(32, 36, device=DEVICE))
mean_ref_np = mean_ref_pts.cpu().numpy()
print(f'Mean ref: {mean_ref_pts.shape}')

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
SCAN_FILE    = 'plank_scaled.npy'   # real scan file
SCALE_FACTOR = 1.0                  # divide raw coords by this
APPLY_FLIP   = False                # flip Y/Z axis before centering

STEP_DEG     = 45   # Euler angle grid step in degrees (30 → 1728 runs, 45 → 512, 10 → 46656)
EULER_AXES   = 'xyz'  # scipy convention for Rotation.from_euler — intrinsic XYZ

CACHE_DIR    = os.path.join(EXP_DIR, '_sweep_cache')   # sweep results saved here
os.makedirs(CACHE_DIR, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

raw = np.load(os.path.join(EXP_DIR, SCAN_FILE))
src_base_np = raw[:, :3].astype(np.float32)
src_base = torch.from_numpy(src_base_np).to(DEVICE)
src_base = src_base / SCALE_FACTOR

if APPLY_FLIP:
    flip = torch.tensor([[1,0,0],[0,1,0],[0,0,-1]], dtype=torch.float32, device=DEVICE)
    src_base = src_base @ flip.T

src_base = src_base - src_base.mean(dim=0, keepdim=True)
src_base_np = src_base.cpu().numpy()

print(f'Scan : {SCAN_FILE}  ({src_base.shape[0]} pts)')
print(f'Range: x=[{src_base_np[:,0].min():.3f},{src_base_np[:,0].max():.3f}]',
      f'y=[{src_base_np[:,1].min():.3f},{src_base_np[:,1].max():.3f}]',
      f'z=[{src_base_np[:,2].min():.3f},{src_base_np[:,2].max():.3f}]')

In [ ]:

# ── Model runner ──────────────────────────────────────────────────────────────
def run_bypass(src_pts_in, ref_pts_in=None):
    """Full forward pass: backbone → transformer → coarse → Sinkhorn → LGR.
    Returns estimated_transform (4×4 tensor) and src_pts_0 / ref_pts_0."""
    if ref_pts_in is None:
        ref_pts_in = mean_ref_pts
    n_ref = ref_pts_in.shape[0]
    n_src = src_pts_in.shape[0]

    concat_pts = torch.cat([ref_pts_in, src_pts_in], dim=0)
    lengths_cpu = torch.tensor([n_ref, n_src], dtype=torch.int64)

    graph = precompute_data_stack_mode(
        concat_pts.cpu(), lengths_cpu,
        model.num_stages, model.init_voxel_size, model.init_radius, model.neighbor_limits,
    )
    for key in ['points', 'lengths', 'neighbors', 'subsampling', 'upsampling']:
        if key in graph:
            graph[key] = [t.to(DEVICE) for t in graph[key]]

    dd = {**graph, 'features': torch.ones(n_ref + n_src, 1, device=DEVICE)}

    with torch.no_grad():
        feats_list = model.backbone(dd['features'], dd)
        feats_c = feats_list[-1]
        feats_f = feats_list[0]

        ref_len_c = dd['lengths'][-1][0].item()
        ref_len_f = dd['lengths'][1][0].item()
        ref_len   = dd['lengths'][0][0].item()

        pts_c = dd['points'][-1];  ref_pts_c = pts_c[:ref_len_c];  src_pts_c = pts_c[ref_len_c:]
        pts_f = dd['points'][1];   ref_pts_f = pts_f[:ref_len_f];  src_pts_f = pts_f[ref_len_f:]
        pts_0 = dd['points'][0];   ref_pts_0 = pts_0[:ref_len];    src_pts_0 = pts_0[ref_len:]

        _, ref_node_masks, ref_knn_idx, ref_knn_masks = point_to_node_partition(
            ref_pts_f, ref_pts_c, model.num_points_in_patch)
        _, src_node_masks, src_knn_idx, src_knn_masks = point_to_node_partition(
            src_pts_f, src_pts_c, model.num_points_in_patch)

        ref_pad_f = torch.cat([ref_pts_f, torch.zeros_like(ref_pts_f[:1])], dim=0)
        src_pad_f = torch.cat([src_pts_f, torch.zeros_like(src_pts_f[:1])], dim=0)
        ref_knn_pts = index_select(ref_pad_f, ref_knn_idx, dim=0)
        src_knn_pts = index_select(src_pad_f, src_knn_idx, dim=0)

        ref_fc_raw = feats_c[:ref_len_c]
        src_fc_raw = feats_c[ref_len_c:]
        ref_feats_f = feats_f[:ref_len_f]
        src_feats_f = feats_f[ref_len_f:]

        ref_fc_out, src_fc_out = model.transformer(
            ref_pts_c.unsqueeze(0), src_pts_c.unsqueeze(0),
            ref_fc_raw.unsqueeze(0), src_fc_raw.unsqueeze(0),
        )
        ref_feats_c = F.normalize(ref_fc_out.squeeze(0), p=2, dim=1)
        src_feats_c = F.normalize(src_fc_out.squeeze(0), p=2, dim=1)

        ref_ci, src_ci, corr_scores_c = model.coarse_matching(
            ref_feats_c, src_feats_c, ref_node_masks, src_node_masks)

        ref_ck_idx = ref_knn_idx[ref_ci];    src_ck_idx = src_knn_idx[src_ci]
        ref_ck_masks = ref_knn_masks[ref_ci]; src_ck_masks = src_knn_masks[src_ci]
        ref_ck_pts   = ref_knn_pts[ref_ci];   src_ck_pts   = src_knn_pts[src_ci]

        ref_pad_ff = torch.cat([ref_feats_f, torch.zeros_like(ref_feats_f[:1])], dim=0)
        src_pad_ff = torch.cat([src_feats_f, torch.zeros_like(src_feats_f[:1])], dim=0)
        ref_ck_feats = index_select(ref_pad_ff, ref_ck_idx, dim=0)
        src_ck_feats = index_select(src_pad_ff, src_ck_idx, dim=0)

        ms = torch.einsum('bnd,bmd->bnm', ref_ck_feats, src_ck_feats) / feats_f.shape[1] ** 0.5
        ms = model.optimal_transport(ms, ref_ck_masks, src_ck_masks)
        if not model.fine_matching.use_dustbin:
            ms = ms[:, :-1, :-1]

        ref_cp, src_cp, cp_scores, est_T = model.fine_matching(
            ref_ck_pts, src_ck_pts, ref_ck_masks, src_ck_masks, ms, corr_scores_c)

    return dict(estimated_transform=est_T, src_pts_0=src_pts_0, ref_pts_0=ref_pts_0)


# ── Metrics ───────────────────────────────────────────────────────────────────
def rot_angle_deg(T):
    R = T[:3, :3].cpu().numpy() if isinstance(T, torch.Tensor) else T[:3, :3]
    return float(np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1.0, 1.0))))

def residual_angle_deg(R_pred_np, R_effective_np):
    """angle(R_pred · R_effective) — should be ~0 if the prediction was perfect.

    For real scans:   R_effective = R_sweep
    For val samples:  R_effective = R_sweep @ R_gt.T
                      because src already carries R_gt, so a perfect model
                      predicts R_gt @ R_sweep⁻¹, and we want
                      angle(R_pred · (R_gt @ R_sweep⁻¹)⁻¹) = angle(R_pred · R_sweep · R_gt.T).
    """
    R_res = R_pred_np @ R_effective_np
    return float(np.degrees(np.arccos(np.clip((np.trace(R_res) - 1) / 2, -1.0, 1.0))))

def nn_dist(A_np, B_np):
    A = torch.from_numpy(A_np).float().to(DEVICE)
    B = torch.from_numpy(B_np).float().to(DEVICE)
    return float(torch.cdist(A, B).min(dim=1).values.mean().item())


# ── Euler grid builder ────────────────────────────────────────────────────────
def build_euler_grid(step_deg, axes='xyz'):
    """Returns list of (alpha, beta, gamma) tuples covering [0, 360) in all axes."""
    vals = np.arange(0, 360, step_deg)
    grid = list(itertools.product(vals, repeat=3))
    return grid, vals


# ── Full sweep ────────────────────────────────────────────────────────────────
def run_euler_sweep(src_base_np_in, ref_pts=None, step_deg=30, axes='xyz',
                    gt_rotation=None, cache_path=None, desc='sweep'):
    """
    Pre-rotates src by each (alpha, beta, gamma) in the grid, runs the model,
    and records residual_angle and nn_dist for each cell.

    gt_rotation : (3, 3) float32 ndarray or None
        The rotation already baked into src_base_np_in (i.e. transform[:3,:3]
        from the val dataset sample).  When provided, the residual is computed
        as angle(R_pred · R_sweep · R_gt.T) so that a perfect prediction gives 0°
        regardless of the starting orientation.  Pass None for real scans
        (equivalent to R_gt = I).

    Returns a dict with:
        grid       : list of (alpha, beta, gamma) tuples
        vals       : 1-D array of angle values used
        residuals  : array of shape (N_grid,) — residual angle in degrees
        nn_dists   : array of shape (N_grid,) — mean NN dist after alignment
        raw_angles : array of shape (N_grid,) — raw predicted R angle
    """
    if cache_path and os.path.exists(cache_path):
        print(f'Loading cached sweep from {cache_path}')
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    if ref_pts is None:
        ref_pts = mean_ref_pts

    grid, vals = build_euler_grid(step_deg, axes)
    n = len(grid)
    residuals  = np.zeros(n, dtype=np.float32)
    nn_dists   = np.zeros(n, dtype=np.float32)
    raw_angles = np.zeros(n, dtype=np.float32)

    src_base_t = torch.from_numpy(src_base_np_in).float().to(DEVICE)
    ref_np = ref_pts.cpu().numpy()

    for i, (a, b, g) in enumerate(tqdm(grid, desc=desc)):
        R_sweep = Rotation.from_euler(axes, [a, b, g], degrees=True).as_matrix().astype(np.float32)
        R_t     = torch.from_numpy(R_sweep).to(DEVICE)
        src_rot = src_base_t @ R_t.T       # pre-rotate source

        res = run_bypass(src_rot, ref_pts)
        T_pred = res['estimated_transform']
        R_pred = T_pred[:3, :3].cpu().numpy()

        # Correct for the GT rotation already present in val src.
        # Real scan: R_effective = R_sweep  (R_gt = I)
        # Val sample: R_effective = R_sweep @ R_gt.T
        if gt_rotation is not None:
            R_effective = R_sweep @ gt_rotation.T
        else:
            R_effective = R_sweep

        residuals[i]  = residual_angle_deg(R_pred, R_effective)
        raw_angles[i] = rot_angle_deg(T_pred)

        src_aln = apply_transform(res['src_pts_0'], T_pred).cpu().numpy()
        ref_0   = res['ref_pts_0'].cpu().numpy()
        nn_dists[i] = nn_dist(src_aln, ref_0)

    result = dict(
        grid=grid, vals=vals, step_deg=step_deg, axes=axes,
        residuals=residuals, nn_dists=nn_dists, raw_angles=raw_angles,
        gt_rotation=gt_rotation,
    )
    if cache_path:
        with open(cache_path, 'wb') as f:
            pickle.dump(result, f)
        print(f'Saved to {cache_path}')
    return result


print('Helpers ready.')


---
## 2 — Single Real Scan Sweep

Run the full Euler grid on **one real scan**.  
Results are cached — re-running the cell loads from disk if the cache exists.

In [ ]:
scan_name  = os.path.splitext(SCAN_FILE)[0]
cache_path = os.path.join(CACHE_DIR, f'{scan_name}_step{STEP_DEG}.pkl')

sweep = run_euler_sweep(
    src_base_np,
    step_deg=STEP_DEG,
    axes=EULER_AXES,
    cache_path=cache_path,
    desc=f'{scan_name} step={STEP_DEG}°',
)

vals   = sweep['vals']
grid   = sweep['grid']
res    = sweep['residuals']
nnd    = sweep['nn_dists']
rawang = sweep['raw_angles']
n_vals = len(vals)

print(f'Grid size      : {len(grid)} cells  ({n_vals} values per axis, step={STEP_DEG}°)')
print(f'Residual angle : min={res.min():.1f}°  max={res.max():.1f}°  mean={res.mean():.1f}°  median={np.median(res):.1f}°')
print(f'NN dist (m)    : min={nnd.min():.4f}  max={nnd.max():.4f}  mean={nnd.mean():.4f}')
print()
print(f'Cells with residual < 30°: {(res < 30).sum()} / {len(res)} ({100*(res < 30).mean():.1f}%)')
print(f'Cells with residual < 15°: {(res < 15).sum()} / {len(res)} ({100*(res < 15).mean():.1f}%)')

---
## 3 — Visualisations

Four complementary views of the same sweep data:

| Plot | What it shows |
|---|---|
| **3a — 1-D marginals** | How sensitive is quality to each axis independently? |
| **3b — 3 pairwise heatmaps** | 2-D failure map for each pair of axes (min over third) |
| **3c — Heatmap slices** | Full grid: one 2-D heatmap per discrete β value |
| **3d — Interactive 3-D scatter** | All 1 728 cells at once; color = residual angle |

In [ ]:
# ── 3a: 1-D marginals ─────────────────────────────────────────────────────────
# For each axis, marginalise (mean + min) over the other two.

# Reshape into (n_vals, n_vals, n_vals) ordered as (alpha, beta, gamma)
res_cube = res.reshape(n_vals, n_vals, n_vals)
nnd_cube = nnd.reshape(n_vals, n_vals, n_vals)

axis_labels = [f'α ({EULER_AXES[0]})', f'β ({EULER_AXES[1]})', f'γ ({EULER_AXES[2]})']
marginalize_over = [(1, 2), (0, 2), (0, 1)]  # axes to reduce when plotting axis 0, 1, 2

fig, axes_plt = plt.subplots(2, 3, figsize=(15, 7))
fig.suptitle(f'1-D marginals — {scan_name}  (step={STEP_DEG}°)', fontsize=13)

for col, (ax_mean, ax_min, red_axes, label) in enumerate(
    zip(axes_plt[0], axes_plt[1], marginalize_over, axis_labels)
):
    mean_1d = res_cube.mean(axis=red_axes)
    min_1d  = res_cube.min(axis=red_axes)

    ax_mean.plot(vals, mean_1d, 'o-', color='steelblue', label='mean residual')
    ax_mean.fill_between(vals,
        res_cube.min(axis=red_axes), res_cube.max(axis=red_axes),
        alpha=0.15, color='steelblue', label='min–max range')
    ax_mean.axhline(30, color='tomato',   linestyle='--', alpha=0.6, label='30°')
    ax_mean.axhline(0,  color='seagreen', linestyle='--', alpha=0.5)
    ax_mean.set_title(f'Mean residual vs {label}\n(marginalised over other 2 axes)')
    ax_mean.set_xlabel(f'{label} (deg)'); ax_mean.set_ylabel('Residual angle (deg)')
    ax_mean.legend(fontsize=8)

    ax_min.plot(vals, min_1d, 's-', color='darkorange', label='best-case (min)')
    ax_min.axhline(30, color='tomato',   linestyle='--', alpha=0.6)
    ax_min.axhline(0,  color='seagreen', linestyle='--', alpha=0.5)
    ax_min.set_title(f'Best-case residual vs {label}')
    ax_min.set_xlabel(f'{label} (deg)'); ax_min.set_ylabel('Min residual angle (deg)')

plt.tight_layout(); plt.show()

In [ ]:
# ── 3b: Three pairwise summary heatmaps ───────────────────────────────────────
# Each 2-D heatmap marginalises the residual with MIN over the third angle.
# Interpretation: for any fixed (axisA, axisB), what is the best residual
# achievable by choosing the third angle optimally?

pairs = [
    (0, 1, 2, axis_labels[0], axis_labels[1], f'min over {axis_labels[2]}'),
    (0, 2, 1, axis_labels[0], axis_labels[2], f'min over {axis_labels[1]}'),
    (1, 2, 0, axis_labels[1], axis_labels[2], f'min over {axis_labels[0]}'),
]

fig, axes_plt = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(f'Pairwise residual heatmaps — {scan_name}  (step={STEP_DEG}°)\n'
             'Color = min residual angle over third axis (dark = model succeeds)',
             fontsize=12)

THRESH = 30   # degrees — above this is a 'failure'
cmap   = plt.cm.RdYlGn_r   # red=bad, green=good
norm   = mcolors.Normalize(vmin=0, vmax=180)

for ax, (a1, a2, a3, lbl1, lbl2, title_suffix) in zip(axes_plt, pairs):
    # min over axis a3  →  shape (n_vals, n_vals)
    heatmap = res_cube.min(axis=a3)          # min over a3
    if a1 > a2:                              # keep row=first, col=second
        heatmap = heatmap.T

    im = ax.imshow(
        heatmap, origin='lower', cmap=cmap, norm=norm,
        extent=[vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2,
                vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2],
        aspect='equal',
    )
    ax.set_xlabel(f'{lbl2} (deg)'); ax.set_ylabel(f'{lbl1} (deg)')
    ax.set_title(f'{lbl1} × {lbl2}\n({title_suffix})')
    ax.set_xticks(vals); ax.set_yticks(vals)
    ax.tick_params(labelsize=6)
    plt.colorbar(im, ax=ax, label='min residual (deg)', fraction=0.046)

plt.tight_layout(); plt.show()

In [ ]:

# ── 3c: Full heatmap slices — one per β value ─────────────────────────────────
# Shows the complete (α, γ) residual map for each fixed β.
# Most detailed view; useful for spotting orientation-specific failure clusters.

ncols = min(n_vals, 6)
nrows = int(np.ceil(n_vals / ncols))
fig, axes_grid = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.2 * nrows))
fig.suptitle(
    f'Residual angle heatmaps — α ({EULER_AXES[0]}) × γ ({EULER_AXES[2]}) per fixed β ({EULER_AXES[1]})\n'
    f'{scan_name}  step={STEP_DEG}°   green contour = {THRESH}° threshold',
    fontsize=12,
)
axes_flat = axes_grid.flatten() if nrows > 1 else np.array(axes_grid).flatten()

im_last = None
for b_idx, beta_val in enumerate(vals):
    ax = axes_flat[b_idx]
    # res_cube shape: (n_alpha, n_beta, n_gamma)
    slice_2d = res_cube[:, b_idx, :]   # (n_alpha, n_gamma)

    im_last = ax.imshow(
        slice_2d, origin='lower', cmap=cmap, norm=norm,
        extent=[vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2,
                vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2],
        aspect='equal',
    )
    ax.contour(vals, vals, slice_2d.T, levels=[THRESH],
               colors=['seagreen'], linewidths=1.2)
    ax.set_title(f'β={beta_val:.0f}°\nmean={slice_2d.mean():.0f}° ok={100*(slice_2d<THRESH).mean():.0f}%',
                 fontsize=9)
    ax.set_xlabel(f'α ({EULER_AXES[0]})', fontsize=7)
    ax.set_ylabel(f'γ ({EULER_AXES[2]})', fontsize=7)
    ax.set_xticks(vals[::2]); ax.set_yticks(vals[::2])
    ax.tick_params(labelsize=6)

# Hide unused subplots
for ax in axes_flat[n_vals:]:
    ax.axis('off')

# Shared colorbar on the right
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.70])
cb = fig.colorbar(im_last, cax=cbar_ax)
cb.set_label('Residual angle (deg)', fontsize=10)
cb.set_ticks([0, 30, 60, 90, 120, 150, 180])

plt.show()


In [ ]:
# ── 3d: Interactive Plotly 3-D scatter ────────────────────────────────────────
# Each sphere = one (α, β, γ) cell; color = residual angle; size = NN dist.
# Hover shows all three angles + metrics.

alphas = np.array([g[0] for g in grid])
betas  = np.array([g[1] for g in grid])
gammas = np.array([g[2] for g in grid])

fig_3d = go.Figure(data=go.Scatter3d(
    x=alphas, y=betas, z=gammas,
    mode='markers',
    marker=dict(
        size=np.clip(nnd * 120 + 3, 3, 12),   # size ∝ NN dist (bigger = worse)
        color=res,
        colorscale='RdYlGn_r',
        cmin=0, cmax=180,
        colorbar=dict(title='Residual (deg)'),
        opacity=0.75,
    ),
    customdata=np.stack([res, nnd, rawang], axis=1),
    hovertemplate=(
        f'α={EULER_AXES[0]}: %{{x:.0f}}°<br>'
        f'β={EULER_AXES[1]}: %{{y:.0f}}°<br>'
        f'γ={EULER_AXES[2]}: %{{z:.0f}}°<br>'
        'residual: %{customdata[0]:.1f}°<br>'
        'NN dist: %{customdata[1]:.4f} m<br>'
        'raw R angle: %{customdata[2]:.1f}°<extra></extra>'
    ),
))
fig_3d.update_layout(
    title=f'3-D Euler sweep — {scan_name}  step={STEP_DEG}°<br>'
          f'Color=residual angle (green=good), Size∝NN dist',
    scene=dict(
        xaxis_title=f'α ({EULER_AXES[0]}) deg',
        yaxis_title=f'β ({EULER_AXES[1]}) deg',
        zaxis_title=f'γ ({EULER_AXES[2]}) deg',
        aspectmode='cube',
    ),
    height=700, margin=dict(l=0, r=0, b=0, t=60),
)
fig_3d.show()

In [ ]:
# ── 3e: Same pairwise heatmaps but colored by NN dist (alignment quality) ─────
# Confirms that residual angle and NN dist tell the same story.

nnd_cube = nnd.reshape(n_vals, n_vals, n_vals)

fig, axes_plt = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(f'NN dist heatmaps — {scan_name}  (step={STEP_DEG}°)\n'
             'Color = min NN dist over third axis (dark = better alignment)', fontsize=12)

cmap_nn = plt.cm.YlOrRd

for ax, (a1, a2, a3, lbl1, lbl2, title_suffix) in zip(axes_plt, pairs):
    heatmap = nnd_cube.min(axis=a3)
    if a1 > a2:
        heatmap = heatmap.T
    im = ax.imshow(
        heatmap, origin='lower', cmap=cmap_nn,
        extent=[vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2,
                vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2],
        aspect='equal',
    )
    ax.set_xlabel(f'{lbl2} (deg)'); ax.set_ylabel(f'{lbl1} (deg)')
    ax.set_title(f'{lbl1} × {lbl2}  ({title_suffix})')
    ax.set_xticks(vals); ax.set_yticks(vals)
    ax.tick_params(labelsize=6)
    plt.colorbar(im, ax=ax, label='min NN dist (m)', fraction=0.046)

plt.tight_layout(); plt.show()

---
## 4 — Validation Sample Sweep

Repeat the same sweep on a **synthetic validation pair** where the ground-truth transform is known.  
This lets us check whether the invariance failure is real-scan-specific or affects synthetic data too.

In [ ]:

# ── Select and load val pair ───────────────────────────────────────────────────
VAL_IDX = 0   # change to try different val pairs

val_dataset = val_loader.dataset
raw_sample  = val_dataset[VAL_IDX]

syn_ref_np = raw_sample['ref_points'].astype(np.float32)
syn_src_np = raw_sample['src_points'].astype(np.float32)

# GT rotation already baked into syn_src (from dataset augmentation).
# transform is 4×4; [:3,:3] is R such that ref ≈ src @ R.T + t.
syn_gt_rotation = raw_sample['transform'][:3, :3].astype(np.float32)

syn_ref = torch.from_numpy(syn_ref_np).float().to(DEVICE)
syn_src = torch.from_numpy(syn_src_np).float().to(DEVICE)

# Centre src (same preprocessing as the real-scan sweep)
syn_src_centered_np = syn_src_np - syn_src_np.mean(axis=0)

print(f'Val pair {VAL_IDX}:')
print(f'  ref {syn_ref_np.shape}  src {syn_src_np.shape}')
print(f'  GT rotation angle: {rot_angle_deg(syn_gt_rotation):.1f}°')

# Quick baseline pass
res_val0 = run_bypass(syn_src, syn_ref)
print(f'  Baseline R angle (no extra sweep rotation): {rot_angle_deg(res_val0["estimated_transform"]):.1f}°')


In [ ]:

val_cache = os.path.join(CACHE_DIR, f'val{VAL_IDX}_step{STEP_DEG}.pkl')

sweep_val = run_euler_sweep(
    syn_src_centered_np,
    ref_pts=syn_ref,
    step_deg=STEP_DEG,
    axes=EULER_AXES,
    gt_rotation=syn_gt_rotation,   # corrects residual for the GT rotation in src
    cache_path=val_cache,
    desc=f'val{VAL_IDX} step={STEP_DEG}°',
)

res_v  = sweep_val['residuals']
nnd_v  = sweep_val['nn_dists']

print(f'Val {VAL_IDX} sweep — {len(res_v)} cells  (GT R angle = {rot_angle_deg(syn_gt_rotation):.1f}°)')
print(f'Residual: min={res_v.min():.1f}°  max={res_v.max():.1f}°  mean={res_v.mean():.1f}°')
print(f'Cells with residual < 30°: {(res_v < 30).sum()} / {len(res_v)} ({100*(res_v < 30).mean():.1f}%)')


In [ ]:
# ── Val: 1-D marginals + pairwise heatmaps side-by-side with real scan ────────

res_v_cube = res_v.reshape(n_vals, n_vals, n_vals)

fig, axes_plt = plt.subplots(2, 3, figsize=(17, 9))
fig.suptitle(f'Residual comparison: real scan vs val pair {VAL_IDX}  (step={STEP_DEG}°)', fontsize=13)

titles_row = [f'α ({EULER_AXES[0]})', f'β ({EULER_AXES[1]})', f'γ ({EULER_AXES[2]})']
reduce_axes = [(1, 2), (0, 2), (0, 1)]

for col, (title, red_ax) in enumerate(zip(titles_row, reduce_axes)):
    ax_real = axes_plt[0, col]
    ax_val  = axes_plt[1, col]

    ax_real.plot(vals, res_cube.mean(axis=red_ax), 'o-', color='tomato',    label=f'{scan_name}')
    ax_real.plot(vals, res_v_cube.mean(axis=red_ax), 's-', color='steelblue', label=f'val {VAL_IDX}')
    ax_real.axhline(30, color='gray', linestyle='--', alpha=0.5)
    ax_real.set_title(f'Mean residual vs {title}')
    ax_real.set_xlabel(f'{title} (deg)'); ax_real.set_ylabel('Residual angle (deg)')
    ax_real.legend(fontsize=8)

    # Success rate (residual < 30°)
    ok_real = (res_cube < 30).mean(axis=red_ax) * 100
    ok_val  = (res_v_cube < 30).mean(axis=red_ax) * 100
    ax_val.bar(vals - STEP_DEG * 0.18, ok_real, width=STEP_DEG * 0.35, color='tomato',    alpha=0.8, label=f'{scan_name}')
    ax_val.bar(vals + STEP_DEG * 0.18, ok_val,  width=STEP_DEG * 0.35, color='steelblue', alpha=0.8, label=f'val {VAL_IDX}')
    ax_val.axhline(50, color='gray', linestyle=':', alpha=0.5)
    ax_val.set_title(f'Success rate (residual<30°) vs {title}')
    ax_val.set_xlabel(f'{title} (deg)'); ax_val.set_ylabel('% cells with residual < 30°')
    ax_val.set_ylim(0, 105); ax_val.legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# ── 3c: Full heatmap slices — one per β value ─────────────────────────────────
# Shows the complete (α, γ) residual map for each fixed β.
# Most detailed view; useful for spotting orientation-specific failure clusters.

ncols = min(n_vals, 6)
nrows = int(np.ceil(n_vals / ncols))
fig, axes_grid = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.2 * nrows))
fig.suptitle(
    f'Residual angle heatmaps — α ({EULER_AXES[0]}) × γ ({EULER_AXES[2]}) per fixed β ({EULER_AXES[1]})\n'
    f'{scan_name}  step={STEP_DEG}°   green contour = {THRESH}° threshold',
    fontsize=12,
)
axes_flat = axes_grid.flatten() if nrows > 1 else np.array(axes_grid).flatten()

im_last = None
for b_idx, beta_val in enumerate(vals):
    ax = axes_flat[b_idx]
    # res_cube shape: (n_alpha, n_beta, n_gamma)
    slice_2d = res_v_cube[:, b_idx, :]   # (n_alpha, n_gamma)

    im_last = ax.imshow(
        slice_2d, origin='lower', cmap=cmap, norm=norm,
        extent=[vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2,
                vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2],
        aspect='equal',
    )
    ax.contour(vals, vals, slice_2d.T, levels=[THRESH],
               colors=['seagreen'], linewidths=1.2)
    ax.set_title(f'β={beta_val:.0f}°\nmean={slice_2d.mean():.0f}° ok={100*(slice_2d<THRESH).mean():.0f}%',
                 fontsize=9)
    ax.set_xlabel(f'α ({EULER_AXES[0]})', fontsize=7)
    ax.set_ylabel(f'γ ({EULER_AXES[2]})', fontsize=7)
    ax.set_xticks(vals[::2]); ax.set_yticks(vals[::2])
    ax.tick_params(labelsize=6)

# Hide unused subplots
for ax in axes_flat[n_vals:]:
    ax.axis('off')

# Shared colorbar on the right
fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.70])
cb = fig.colorbar(im_last, cax=cbar_ax)
cb.set_label('Residual angle (deg)', fontsize=10)
cb.set_ticks([0, 30, 60, 90, 120, 150, 180])

plt.show()


---
## 5 — Multi-Sample Comparison

Run the sweep for **N val pairs** and aggregate to check whether failure modes are consistent across samples  
(systematic → model bias) or sample-specific (→ individual scan preprocessing issue).

Set `N_VAL_PAIRS` and `N_REAL_SCANS` below.  
Results are cached per sample; re-run is cheap after the first time.

In [ ]:

# ── CONFIG ─────────────────────────────────────────────────────────────────────
N_VAL_PAIRS   = 10    # number of val pairs to sweep
REAL_SCANS_DIR = os.path.join(ROOT_DIR, 'planck_scans', 'npy_scaled_uniform')
# ─────────────────────────────────────────────────────────────────────────────

all_sweeps = {}   # key → sweep result dict

# Real scans — load all .npy files from the folder
scan_files = sorted(f for f in os.listdir(REAL_SCANS_DIR) if f.endswith('.npy'))
print(f'Found {len(scan_files)} real scans in {REAL_SCANS_DIR}')

for fname in scan_files:
    raw = np.load(os.path.join(REAL_SCANS_DIR, fname))
    pts = raw[:, :3].astype(np.float32)
    pts = pts - pts.mean(axis=0)
    key = os.path.splitext(fname)[0]
    cache = os.path.join(CACHE_DIR, f'{key}_step{STEP_DEG}.pkl')
    all_sweeps[key] = run_euler_sweep(
        pts, step_deg=STEP_DEG, axes=EULER_AXES,
        gt_rotation=None, cache_path=cache, desc=key,
    )

# Val pairs — pass gt_rotation so residuals are comparable to real scans
for i in range(N_VAL_PAIRS):
    raw_i   = val_dataset[i]
    src_i   = raw_i['src_points'].astype(np.float32)
    ref_i   = torch.from_numpy(raw_i['ref_points'].astype(np.float32)).float().to(DEVICE)
    R_gt_i  = raw_i['transform'][:3, :3].astype(np.float32)
    src_i_c = src_i - src_i.mean(axis=0)
    key = f'val_{i:02d}'
    cache = os.path.join(CACHE_DIR, f'{key}_step{STEP_DEG}.pkl')
    all_sweeps[key] = run_euler_sweep(
        src_i_c, ref_pts=ref_i, step_deg=STEP_DEG, axes=EULER_AXES,
        gt_rotation=R_gt_i, cache_path=cache, desc=key,
    )

print(f'\nLoaded {len(all_sweeps)} sample sweeps total.')


In [ ]:
# ── Summary table ─────────────────────────────────────────────────────────────
print(f'{"Sample":25s}  {"Mean res":>10s}  {"Median res":>12s}  {"<30° %":>8s}  {"<15° %":>8s}  {"Mean NN":>9s}')
print('-' * 80)
for key, sw in all_sweeps.items():
    r = sw['residuals']
    n = sw['nn_dists']
    print(f'{key:25s}  {r.mean():10.1f}°  {np.median(r):12.1f}°  '
          f'{100*(r<30).mean():8.1f}%  {100*(r<15).mean():8.1f}%  {n.mean():9.4f} m')

# ── Stacked mean-residual 1-D marginals comparison ─────────────────────────────
fig, axes_plt = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(f'Mean residual 1-D marginals — all samples  (step={STEP_DEG}°)', fontsize=13)

colors = plt.cm.tab10(np.linspace(0, 1, len(all_sweeps)))

for col, (red_ax, lbl) in enumerate(zip(reduce_axes, titles_row)):
    ax = axes_plt[col]
    for (key, sw), color in zip(all_sweeps.items(), colors):
        cube = sw['residuals'].reshape(n_vals, n_vals, n_vals)
        linestyle = '-' if key.startswith('val') else '--'
        ax.plot(vals, cube.mean(axis=red_ax), linestyle=linestyle,
                marker='o', markersize=4, color=color, label=key, linewidth=1.4)
    ax.axhline(30, color='gray', linestyle=':', alpha=0.5)
    ax.set_title(f'vs {lbl}')
    ax.set_xlabel(f'{lbl} (deg)'); ax.set_ylabel('Mean residual (deg)')
    if col == 0:
        ax.legend(fontsize=7, ncol=2)

plt.tight_layout(); plt.show()

# ── Consensus failure map: fraction of samples with residual > 30° ─────────────
fail_cubes = []
for sw in all_sweeps.values():
    fail_cubes.append((sw['residuals'] > 30).reshape(n_vals, n_vals, n_vals).astype(float))
consensus_fail = np.stack(fail_cubes, axis=0).mean(axis=0)   # (n_vals, n_vals, n_vals)

# Pairwise heatmaps of consensus failure (mean over third axis)
fig, axes_plt = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle(f'Consensus failure rate (fraction of samples with residual > 30°)\n'
             f'step={STEP_DEG}°  — dark red = fails on all samples', fontsize=12)

for ax, (a1, a2, a3, lbl1, lbl2, title_suffix) in zip(axes_plt, pairs):
    hmap = consensus_fail.mean(axis=a3)
    if a1 > a2:
        hmap = hmap.T
    im = ax.imshow(
        hmap, origin='lower', cmap='RdYlGn_r', vmin=0, vmax=1,
        extent=[vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2,
                vals[0] - STEP_DEG/2, vals[-1] + STEP_DEG/2],
        aspect='equal',
    )
    ax.set_xlabel(f'{lbl2} (deg)'); ax.set_ylabel(f'{lbl1} (deg)')
    ax.set_title(f'{lbl1} × {lbl2}  ({title_suffix})')
    ax.set_xticks(vals); ax.set_yticks(vals)
    ax.tick_params(labelsize=6)
    plt.colorbar(im, ax=ax, label='failure rate (0–1)', fraction=0.046)

plt.tight_layout(); plt.show()

---
## 6 — Full Real-Scan Heatmap Grid

Two views over all 20 planck scans:

| Plot | What it shows |
|---|---|
| **6a — Aggregated β-slices** | Mean residual per (α, γ) cell across all scans, one panel per β — same format as §3c but collapsed over subjects |
| **6b — Per-scan overview** | One panel per scan (4×5 grid), each showing the best-case (α×γ) map (min over β) so failure clusters are comparable at a glance |

In [ ]:

# ── 6a: Aggregated β-slice heatmaps across all real scans ─────────────────────
# Stack residual cubes for every planck scan, compute mean and std per cell,
# then plot the same β-slice grid as §3c.

real_keys = [k for k in all_sweeps if not k.startswith('val_')]
print(f'Real scans included: {len(real_keys)}')

real_cubes = np.stack(
    [all_sweeps[k]['residuals'].reshape(n_vals, n_vals, n_vals) for k in real_keys],
    axis=0,
)   # (n_scans, n_alpha, n_beta, n_gamma)

mean_cube = real_cubes.mean(axis=0)   # (n_alpha, n_beta, n_gamma)
std_cube  = real_cubes.std(axis=0)

ncols = min(n_vals, 6)
nrows = int(np.ceil(n_vals / ncols))

# ── Mean ──────────────────────────────────────────────────────────────────────
fig, axes_grid = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.2 * nrows))
fig.suptitle(
    f'Mean residual — α ({EULER_AXES[0]}) × γ ({EULER_AXES[2]}) per fixed β ({EULER_AXES[1]})\n'
    f'Averaged over {len(real_keys)} real scans   step={STEP_DEG}°   '
    f'green contour = {THRESH}° threshold',
    fontsize=12,
)
axes_flat = axes_grid.flatten() if nrows > 1 else np.array(axes_grid).flatten()
im_last = None
for b_idx, beta_val in enumerate(vals):
    ax = axes_flat[b_idx]
    slice_2d = mean_cube[:, b_idx, :]   # (n_alpha, n_gamma)
    im_last = ax.imshow(
        slice_2d, origin='lower', cmap=cmap, norm=norm,
        extent=[vals[0]-STEP_DEG/2, vals[-1]+STEP_DEG/2,
                vals[0]-STEP_DEG/2, vals[-1]+STEP_DEG/2],
        aspect='equal',
    )
    ax.contour(vals, vals, slice_2d.T, levels=[THRESH],
               colors=['seagreen'], linewidths=1.2)
    ok_pct = 100 * (real_cubes[:, :, b_idx, :] < THRESH).mean()
    ax.set_title(f'β={beta_val:.0f}°\nmean={slice_2d.mean():.0f}° ok={ok_pct:.0f}%', fontsize=9)
    ax.set_xlabel(f'α ({EULER_AXES[0]})', fontsize=7)
    ax.set_ylabel(f'γ ({EULER_AXES[2]})', fontsize=7)
    ax.set_xticks(vals[::2]); ax.set_yticks(vals[::2])
    ax.tick_params(labelsize=6)

for ax in axes_flat[n_vals:]:
    ax.axis('off')

fig.subplots_adjust(right=0.88)
cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.70])
cb = fig.colorbar(im_last, cax=cbar_ax)
cb.set_label('Mean residual angle (deg)', fontsize=10)
cb.set_ticks([0, 30, 60, 90, 120, 150, 180])
plt.show()

# ── Std ───────────────────────────────────────────────────────────────────────
fig2, axes_grid2 = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.2 * nrows))
fig2.suptitle(
    f'Std of residual — α ({EULER_AXES[0]}) × γ ({EULER_AXES[2]}) per fixed β ({EULER_AXES[1]})\n'
    f'Across {len(real_keys)} real scans   step={STEP_DEG}°   (high std = scan-dependent failure)',
    fontsize=12,
)
axes_flat2 = axes_grid2.flatten() if nrows > 1 else np.array(axes_grid2).flatten()
im_std = None
std_norm = mcolors.Normalize(vmin=0, vmax=60)
for b_idx, beta_val in enumerate(vals):
    ax = axes_flat2[b_idx]
    slice_std = std_cube[:, b_idx, :]
    im_std = ax.imshow(
        slice_std, origin='lower', cmap='YlOrBr', norm=std_norm,
        extent=[vals[0]-STEP_DEG/2, vals[-1]+STEP_DEG/2,
                vals[0]-STEP_DEG/2, vals[-1]+STEP_DEG/2],
        aspect='equal',
    )
    ax.set_title(f'β={beta_val:.0f}°\nstd={slice_std.mean():.0f}°', fontsize=9)
    ax.set_xlabel(f'α ({EULER_AXES[0]})', fontsize=7)
    ax.set_ylabel(f'γ ({EULER_AXES[2]})', fontsize=7)
    ax.set_xticks(vals[::2]); ax.set_yticks(vals[::2])
    ax.tick_params(labelsize=6)

for ax in axes_flat2[n_vals:]:
    ax.axis('off')

fig2.subplots_adjust(right=0.88)
cbar_ax2 = fig2.add_axes([0.90, 0.15, 0.015, 0.70])
cb2 = fig2.colorbar(im_std, cax=cbar_ax2)
cb2.set_label('Std of residual angle (deg)', fontsize=10)
plt.show()


In [ ]:

# ── 6b: Per-scan overview — 4×5 grid, one panel per real scan ─────────────────
# Each panel shows the best-case (α × γ) map (min over all β values) for that scan.
# Consistent dark patches across scans = systematic model blind spot.

n_scans = len(real_keys)
n_cols_s = 5
n_rows_s = int(np.ceil(n_scans / n_cols_s))

fig, axes_s = plt.subplots(n_rows_s, n_cols_s,
                            figsize=(3.8 * n_cols_s, 3.4 * n_rows_s))
fig.suptitle(
    f'Per-scan best-case residual — α ({EULER_AXES[0]}) × γ ({EULER_AXES[2]}), min over β ({EULER_AXES[1]})\n'
    f'{n_scans} planck scans   step={STEP_DEG}°   green contour = {THRESH}°',
    fontsize=12,
)
axes_s_flat = axes_s.flatten() if n_rows_s > 1 else np.array(axes_s).flatten()
im_s = None
for idx, key in enumerate(real_keys):
    ax = axes_s_flat[idx]
    cube = all_sweeps[key]['residuals'].reshape(n_vals, n_vals, n_vals)
    # min over β axis (axis=1) → (n_alpha, n_gamma)
    best_ag = cube.min(axis=1)
    im_s = ax.imshow(
        best_ag, origin='lower', cmap=cmap, norm=norm,
        extent=[vals[0]-STEP_DEG/2, vals[-1]+STEP_DEG/2,
                vals[0]-STEP_DEG/2, vals[-1]+STEP_DEG/2],
        aspect='equal',
    )
    ax.contour(vals, vals, best_ag.T, levels=[THRESH],
               colors=['seagreen'], linewidths=1.0)
    ok_pct = 100 * (best_ag < THRESH).mean()
    ax.set_title(f'{key}\nok={ok_pct:.0f}% | mean={best_ag.mean():.0f}°', fontsize=8)
    ax.set_xlabel(f'α', fontsize=7); ax.set_ylabel(f'γ', fontsize=7)
    ax.set_xticks(vals[::2]); ax.set_yticks(vals[::2])
    ax.tick_params(labelsize=5)

for ax in axes_s_flat[n_scans:]:
    ax.axis('off')

fig.subplots_adjust(right=0.88)
cbar_ax_s = fig.add_axes([0.90, 0.15, 0.012, 0.70])
cb_s = fig.colorbar(im_s, cax=cbar_ax_s)
cb_s.set_label('Min residual angle over β (deg)', fontsize=9)
cb_s.set_ticks([0, 30, 60, 90, 120, 150, 180])
plt.show()

# Print ranked summary table
print(f'\n{"Scan":20s}  {"Mean res":>10s}  {"<30% cells":>12s}  {"Best cell":>10s}')
print('-' * 60)
ranked = sorted(real_keys, key=lambda k: all_sweeps[k]['residuals'].mean())
for k in ranked:
    r = all_sweeps[k]['residuals']
    print(f'{k:20s}  {r.mean():9.1f}°  {100*(r<30).mean():11.1f}%  {r.min():9.1f}°')


---
### 6c — Real Scans vs Validation: Separate β-slice grids

To compare heatmaps cell-by-cell, both cubes must be indexed by the **same actual face orientation**.

- Real scans: each cell `R_sweep` already is the face orientation — no change needed.
- Val samples: the face was at `R_sweep · R_gt` (sweep on top of baked-in GT rotation).
  We re-index val results by projecting each `(R_sweep, residual)` pair onto its actual
  global orientation `R_sweep · R_gt`, converting to Euler angles, and accumulating into
  the nearest output bin.  No re-running needed — works on cached sweep results.

Bins with multiple inputs are averaged; bins with no inputs are NaN (shown as grey).

A **difference map** (scans − val) is shown last:
- Red → scans fail more than val at that face orientation → domain gap
- Blue → val fails more than scans → rotation correction over-compensates or val-specific issue

In [ ]:
# ── β-slice grid plot helper (used by 6c) ─────────────────────────────────────────────
def plot_beta_slices(cube, title, cmap_use, norm_use,
                     n_vals, vals, step_deg, thresh, euler_axes,
                     extra_cube=None, extra_label='',
                     cbar_label='Residual angle (deg)'):
    """Plot an (n,n,n) cube as a grid of α×γ heatmaps, one per β value.

    NaN cells show grey. Solid green contour = `cube` at `thresh`;
    white dashed contour = `extra_cube` at `thresh` (if given).
    x = α, y = γ on every panel.
    """
    ncols = min(n_vals, 6)
    nrows = int(np.ceil(n_vals / ncols))
    fig, axes_grid = plt.subplots(nrows, ncols, figsize=(3.5 * ncols, 3.2 * nrows))
    fig.suptitle(title, fontsize=12)
    axes_flat = np.atleast_1d(axes_grid).flatten()
    extent = [vals[0]-step_deg/2, vals[-1]+step_deg/2,
              vals[0]-step_deg/2, vals[-1]+step_deg/2]
    im_last = None
    for b_idx, beta_val in enumerate(vals):
        ax = axes_flat[b_idx]
        slice_2d = np.ma.masked_invalid(cube[:, b_idx, :].T)   # (γ, α): x=α, y=γ
        ax.set_facecolor('lightgrey')                          # shows through NaN cells
        im_last = ax.imshow(slice_2d, origin='lower', cmap=cmap_use,
                            norm=norm_use, extent=extent, aspect='equal')
        if slice_2d.count() > 0:
            ax.contour(vals, vals, slice_2d, levels=[thresh],
                       colors=['seagreen'], linewidths=1.2)
        if extra_cube is not None:
            extra_2d = np.ma.masked_invalid(extra_cube[:, b_idx, :].T)
            if extra_2d.count() > 0:
                ax.contour(vals, vals, extra_2d, levels=[thresh],
                           colors=['white'], linestyles='dashed', linewidths=1.2)
        if slice_2d.count() > 0:
            ok_pct = 100 * (slice_2d < thresh).mean()
            stats = f'mean={slice_2d.mean():.0f}° ok={ok_pct:.0f}%'
        else:
            stats = 'no data'
        ax.set_title(f'β={beta_val:.0f}°\n{stats}', fontsize=9)
        ax.set_xlabel(f'α ({euler_axes[0]})', fontsize=7)
        ax.set_ylabel(f'γ ({euler_axes[2]})', fontsize=7)
        ax.set_xticks(vals[::2]); ax.set_yticks(vals[::2])
        ax.tick_params(labelsize=6)
    for ax in axes_flat[n_vals:]:
        ax.axis('off')
    fig.subplots_adjust(right=0.88)
    cbar_ax = fig.add_axes([0.90, 0.15, 0.015, 0.70])
    cb = fig.colorbar(im_last, cax=cbar_ax)
    cb.set_label(cbar_label, fontsize=10)
    plt.show()


In [ ]:

# ── Re-index helper ───────────────────────────────────────────────────────────
def reindex_by_global_rotation(sweep_result, axes):
    """
    Re-index a val sweep cube so cells correspond to actual face orientations.

    For each sweep cell R_sweep the face was at R_eff = R_sweep @ R_gt.
    We convert R_eff to Euler angles, snap to the nearest grid bin, and
    accumulate residuals there.  Returns a cube of the same shape as the
    original, with NaN where no input mapped to that bin.
    """
    grid      = sweep_result['grid']
    vals      = sweep_result['vals']
    residuals = sweep_result['residuals']
    R_gt      = sweep_result['gt_rotation']   # (3,3) or None
    step_deg  = sweep_result['step_deg']
    n         = len(vals)

    # If no GT rotation (real scan), return reshaped cube unchanged
    if R_gt is None:
        return residuals.reshape(n, n, n).copy()

    accum = np.full((n, n, n), 0.0, dtype=np.float64)
    count = np.zeros((n, n, n), dtype=np.int32)

    for idx, (a, b, g) in enumerate(grid):
        R_sweep = Rotation.from_euler(axes, [a, b, g], degrees=True).as_matrix()
        R_eff   = R_sweep @ R_gt                          # actual face orientation
        angles  = Rotation.from_matrix(R_eff).as_euler(axes, degrees=True) % 360

        # Snap each angle to nearest bin index
        ia = int(np.argmin(np.abs(vals - angles[0])))
        ib = int(np.argmin(np.abs(vals - angles[1])))
        ig = int(np.argmin(np.abs(vals - angles[2])))

        accum[ia, ib, ig] += residuals[idx]
        count[ia, ib, ig] += 1

    out = np.where(count > 0, accum / count, np.nan)
    coverage = (count > 0).mean() * 100
    print(f'  Re-indexed: {coverage:.0f}% of bins filled  '
          f'(avg {count[count>0].mean():.1f} inputs/bin)')
    return out


# ── Build mean cubes ──────────────────────────────────────────────────────────
real_keys = [k for k in all_sweeps if not k.startswith('val_')]
val_keys  = [k for k in all_sweeps if     k.startswith('val_')]

print('Real scans (no re-indexing needed):')
real_cubes_arr = np.stack(
    [all_sweeps[k]['residuals'].reshape(n_vals, n_vals, n_vals) for k in real_keys], axis=0)
scan_mean_cube = real_cubes_arr.mean(axis=0)

print('\nVal pairs (re-indexing by R_sweep · R_gt):')
val_reindexed = []
for k in val_keys:
    print(f'  {k}', end='')
    cube = reindex_by_global_rotation(all_sweeps[k], EULER_AXES)
    val_reindexed.append(cube)

val_stack     = np.stack(val_reindexed, axis=0)           # (n_val, n_a, n_b, n_g)
val_mean_cube = np.nanmean(val_stack, axis=0)             # NaN-safe mean
val_coverage  = np.isfinite(val_mean_cube).mean() * 100

diff_cube = scan_mean_cube - val_mean_cube                # NaN propagates where val has no data

print(f'\nScan mean residual : {scan_mean_cube.mean():.1f}°')
print(f'Val  mean residual : {np.nanmean(val_mean_cube):.1f}°  (coverage={val_coverage:.0f}% of bins)')
print(f'Diff range         : [{np.nanmin(diff_cube):.1f}°, {np.nanmax(diff_cube):.1f}°]')


# ── Plot all three grids ──────────────────────────────────────────────────────
plot_beta_slices(
    scan_mean_cube,
    title=(f'Real scans ({len(real_keys)}) — mean residual per β-slice\n'
           f'green = scan {THRESH}° boundary   white dashed = val boundary'),
    cmap_use=cmap, norm_use=norm,
    n_vals=n_vals, vals=vals, step_deg=STEP_DEG, thresh=THRESH,
    euler_axes=EULER_AXES,
    extra_cube=val_mean_cube, extra_label='val',
)

plot_beta_slices(
    val_mean_cube,
    title=(f'Validation ({len(val_keys)} pairs, re-indexed by R_sweep·R_gt) — mean residual\n'
           f'grey = no val data in bin   white dashed = scan boundary'),
    cmap_use=cmap, norm_use=norm,
    n_vals=n_vals, vals=vals, step_deg=STEP_DEG, thresh=THRESH,
    euler_axes=EULER_AXES,
    extra_cube=scan_mean_cube, extra_label='scan',
)

diff_norm = mcolors.TwoSlopeNorm(vmin=-90, vcenter=0, vmax=90)
plot_beta_slices(
    diff_cube,
    title=(f'Difference: scans − val  (red = scans worse → domain gap, blue = val worse)\n'
           f'grey = bins where val has no data after re-indexing'),
    cmap_use='RdBu_r', norm_use=diff_norm,
    n_vals=n_vals, vals=vals, step_deg=STEP_DEG, thresh=THRESH,
    euler_axes=EULER_AXES,
    extra_cube=None,
)


---
## 7 — Training / Validation Rotation Distribution

**Question:** Are the orientations where the model fails actually under-represented in training?

### What the dataset does

`generate_random_view` samples `R` via QR decomposition of a random Gaussian matrix — this is
**Haar-uniform over SO(3)**, meaning every orientation is equally likely.

### Why Euler histograms still look non-uniform

Projecting a uniform SO(3) distribution onto Euler angles is **not flat**.
For intrinsic XYZ, the marginal densities are:
- α (x), γ (z): uniform on [0°, 360°)
- **β (y): proportional to |sin(β)|** — peaks at 90°, goes to zero at 0° and 180°

So an apparent "gap" near β ≈ 0° or 180° in the histogram is the **expected geometry of the parameterization**, not a coverage gap in SO(3).

The §7c **overlay** is still the right diagnostic: if the model fails more in the |sin(β)|-sparse region near β=0°, it could mean the model wasn't exposed to enough near-upright faces — but we must compare against the correct expected density, not a flat uniform line.

The training also applies an **additional augmentation rotation** (`random_sample_rotation`) on top of the dataset rotation at load time, which compounds the coverage further toward uniform SO(3).

In [ ]:

# ── 7a: Extract rotations from train + val datasets ───────────────────────────

train_loader, val_loader_rot, _ = train_valid_data_loader(cfg, distributed=False)
train_dataset = train_loader.dataset

def collect_euler_angles(dataset, axes, max_samples=None, desc=''):
    """Iterate dataset, extract transform[:3,:3], convert to Euler angles."""
    n = len(dataset) if max_samples is None else min(len(dataset), max_samples)
    euler_angles = np.zeros((n, 3), dtype=np.float32)
    for i in tqdm(range(n), desc=desc or f'extracting ({n} samples)'):
        sample = dataset[i]
        R = sample['transform'][:3, :3]
        # Convert to Euler angles in [0, 360) using the same convention as the sweep.
        # scipy returns angles in [-180, 180]; map to [0, 360).
        angles = Rotation.from_matrix(R).as_euler(axes, degrees=True)
        euler_angles[i] = angles % 360
    return euler_angles

train_eulers = collect_euler_angles(train_dataset, EULER_AXES, desc='train')
val_eulers   = collect_euler_angles(val_loader_rot.dataset, EULER_AXES, desc='val')

print(f'Train samples : {len(train_eulers)}')
print(f'Val samples   : {len(val_eulers)}')
for j, ax_lbl in enumerate(EULER_AXES):
    print(f'  {ax_lbl}: train [{train_eulers[:,j].min():.1f}°, {train_eulers[:,j].max():.1f}°]'
          f'  val [{val_eulers[:,j].min():.1f}°, {val_eulers[:,j].max():.1f}°]')


In [ ]:

# ── 7b: 1-D marginal histograms with correct SO(3) reference density ──────────
#
# For intrinsic XYZ Euler angles, a Haar-uniform SO(3) distribution gives:
#   α (x-axis) : uniform  →  reference = 1/360
#   β (y-axis) : Tait-Bryan middle angle lives in [-90°, 90°] with density ∝ cos(β);
#                wrapped to [0°, 360°) this is ∝ max(cos β, 0) — peaks at 0°/360°,
#                identically ZERO on (90°, 270°).
#   γ (z-axis) : uniform  →  reference = 1/360
#
# (|sin β| would be the reference for proper-Euler conventions like ZYZ, not xyz.)
# The empty β region between 90° and 270° is expected from the parameterisation,
# NOT a coverage hole in SO(3).

bins = np.arange(0, 361, STEP_DEG)
bin_centres = 0.5 * (bins[:-1] + bins[1:])

# Expected SO(3) density in each bin for each axis (intrinsic XYZ).
# α and γ: uniform.  β: |sin(β)| normalised so the histogram sums to 1.
def expected_so3_density(bin_centres_deg, axis_index):
    """Expected histogram density for Haar-uniform SO(3), axis_index 0/1/2 = x/y/z."""
    if axis_index == 1:   # β — cos-weighted on [-90°, 90°], zero elsewhere (xyz convention)
        weights = np.clip(np.cos(np.radians(bin_centres_deg)), 0.0, None)
        return weights / weights.sum()
    else:                 # α, γ — uniform
        return np.ones_like(bin_centres_deg) / len(bin_centres_deg)

fig, axes_plt = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Euler angle marginal distributions — training vs validation\n'
             '(dashed = expected density for Haar-uniform SO(3))', fontsize=12)

for j, (ax, ax_lbl) in enumerate(zip(axes_plt, list(EULER_AXES))):
    train_counts, _ = np.histogram(train_eulers[:, j], bins=bins)
    val_counts,   _ = np.histogram(val_eulers[:,   j], bins=bins)
    train_dens = train_counts / train_counts.sum()
    val_dens   = val_counts   / val_counts.sum()

    ax.bar(bin_centres - STEP_DEG*0.22, train_dens, width=STEP_DEG*0.42,
           alpha=0.75, color='steelblue', label=f'train (n={len(train_eulers)})')
    ax.bar(bin_centres + STEP_DEG*0.22, val_dens,   width=STEP_DEG*0.42,
           alpha=0.75, color='tomato',    label=f'val (n={len(val_eulers)})')

    ref = expected_so3_density(bin_centres, j)
    ax.step(bins[:-1], ref, where='post', color='black',
            linestyle='--', linewidth=1.5, label='SO(3) expected')

    ax.set_xlabel(f'{ax_lbl} angle (deg)')
    ax.set_ylabel('fraction of samples')
    ax.set_title(f'Axis {ax_lbl}' + (' (|cos(β)| expected)' if j == 1 else ' (uniform expected)'))
    ax.set_xlim(0, 360)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:

# ── 7c: Density heatmaps + overlay with sweep residuals ───────────────────────
# Build a 3-D density histogram using the same bins as the sweep grid.
# Then show the same three pairwise projections as §3b, but with:
#   - background color = training sample density (marginalised over third axis)
#   - red contour     = mean residual > THRESH (from §6a aggregated cube)
#
# Side-by-side: density | residual | overlay

bins_edges = np.append(vals - STEP_DEG / 2, vals[-1] + STEP_DEG / 2)  # n_vals+1 edges

# 3-D histogram: (n_alpha, n_beta, n_gamma)
train_hist, _ = np.histogramdd(train_eulers, bins=[bins_edges, bins_edges, bins_edges])
val_hist,   _ = np.histogramdd(val_eulers,   bins=[bins_edges, bins_edges, bins_edges])

# Normalise to density (sum=1)
train_density = train_hist / train_hist.sum()
val_density   = val_hist   / val_hist.sum()

pair_configs = [
    (0, 1, 2, f'α ({EULER_AXES[0]})', f'β ({EULER_AXES[1]})', f'γ ({EULER_AXES[2]})'),
    (0, 2, 1, f'α ({EULER_AXES[0]})', f'γ ({EULER_AXES[2]})', f'β ({EULER_AXES[1]})'),
    (1, 2, 0, f'β ({EULER_AXES[1]})', f'γ ({EULER_AXES[2]})', f'α ({EULER_AXES[0]})'),
]

for source_label, density_cube in [('train', train_density), ('val', val_density)]:
    fig, axes_plt = plt.subplots(2, 3, figsize=(17, 10))
    fig.suptitle(
        f'Rotation density ({source_label}) vs mean sweep residual\n'
        f'Top: density   Bottom: overlay (density + residual>{THRESH}° contour in red)',
        fontsize=12,
    )

    for col, (a1, a2, a3, lbl1, lbl2, lbl3) in enumerate(pair_configs):
        ax_dens    = axes_plt[0, col]
        ax_overlay = axes_plt[1, col]

        # Marginalise density by summing over third axis
        dens_2d = density_cube.sum(axis=a3)
        if a1 > a2:
            dens_2d = dens_2d.T

        # Marginalise residual by mean over third axis (from §6a mean_cube)
        res_2d = mean_cube.mean(axis=a3)
        if a1 > a2:
            res_2d = res_2d.T

        ext = [vals[0]-STEP_DEG/2, vals[-1]+STEP_DEG/2,
               vals[0]-STEP_DEG/2, vals[-1]+STEP_DEG/2]

        # Density
        im_d = ax_dens.imshow(dens_2d, origin='lower', cmap='Blues', aspect='equal', extent=ext)
        ax_dens.set_title(f'{lbl1} × {lbl2}\n(sum over {lbl3})')
        ax_dens.set_xlabel(f'{lbl2} (deg)'); ax_dens.set_ylabel(f'{lbl1} (deg)')
        ax_dens.set_xticks(vals); ax_dens.set_yticks(vals)
        ax_dens.tick_params(labelsize=6)
        plt.colorbar(im_d, ax=ax_dens, label='density', fraction=0.046)

        # Overlay: density background + residual contour
        ax_overlay.imshow(dens_2d, origin='lower', cmap='Blues', aspect='equal', extent=ext)
        # Residual heatmap at 50% transparency on top
        ax_overlay.imshow(res_2d, origin='lower', cmap=cmap, norm=norm,
                          alpha=0.45, aspect='equal', extent=ext)
        # Hard contour at THRESH
        ax_overlay.contour(vals, vals, res_2d.T, levels=[THRESH],
                           colors=['red'], linewidths=1.5, linestyles='--')
        ax_overlay.set_title(f'Overlay: density (blue) + residual (RdYlGn)\nred dashed = {THRESH}° boundary')
        ax_overlay.set_xlabel(f'{lbl2} (deg)'); ax_overlay.set_ylabel(f'{lbl1} (deg)')
        ax_overlay.set_xticks(vals); ax_overlay.set_yticks(vals)
        ax_overlay.tick_params(labelsize=6)

    plt.tight_layout()
    plt.show()


---
## 8 · Hopf sweep — duplicate-free, gimbal-free SO(3) coverage

Replaces the Euler-grid sweep machinery (§3–§6), which has three structural problems:

1. **Double cover** — (α, β, γ) and (α+180°, 180°−β, γ+180°) are the *same rotation*, and with a 45° step both land on the grid: 512 cells test only ~256 distinct orientations, each twice.
2. **Gimbal lock** — at β = 90°/270° a whole 8×8 α×γ panel collapses to a near-1-D family of rotations.
3. **Non-Haar weighting** — uniform Euler cells oversample orientations near gimbal lock, biasing mean / ok% statistics.

**Approach:** Haar measure on SO(3) factors *exactly* as uniform(S²) × uniform(roll) (Hopf fibration). We take Fibonacci-sphere directions × evenly spaced, phase-staggered rolls → N **distinct**, near-uniformly spaced rotations. Plain averages over cells are then unbiased, and plots become sphere maps ("where does the face's +z axis point"), one panel per roll bin — no phantom duplicate panels, no degenerate slices.

Val sweeps are compared to scans by **quaternion geodesic nearest-neighbour re-indexing** — no unreachable bins, no per-axis Euler snapping.

⚠ Requires running the model sweeps once for the new rotation set — cached as `*_hopf{D}x{R}.pkl` in `_sweep_cache/`.


In [ ]:

# ── 8a: Hopf rotation set — Fibonacci directions × uniform roll ───────────────
# Each rotation = "point the face's canonical +z axis along direction v, then
# roll θ about it".  Haar-uniform ⇔ v uniform on S² and θ uniform on the fiber.

GOLDEN = (1 + 5 ** 0.5) / 2

def fibonacci_sphere(n):
    """n near-uniform unit vectors on S² (golden-angle spiral)."""
    i = np.arange(n) + 0.5
    polar = np.arccos(1 - 2 * i / n)
    azim  = 2 * np.pi * GOLDEN * i
    return np.stack([np.sin(polar) * np.cos(azim),
                     np.sin(polar) * np.sin(azim),
                     np.cos(polar)], axis=1)

def rot_align_z_to(v):
    """Shortest-arc rotation taking +z to unit vector v."""
    z = np.array([0.0, 0.0, 1.0])
    c = float(z @ v)
    if c > 1 - 1e-12:
        return np.eye(3)
    if c < -1 + 1e-12:
        return np.diag([1.0, -1.0, -1.0])          # 180° about x
    axis = np.cross(z, v)
    axis /= np.linalg.norm(axis)
    return Rotation.from_rotvec(axis * np.arccos(np.clip(c, -1.0, 1.0))).as_matrix()

def build_hopf_rotation_set(n_dirs, n_rolls):
    """n_dirs × n_rolls distinct rotations, near-uniform in Haar measure.
    Roll phases are staggered per direction (golden-ratio offset) so fibers
    don't share the same roll origin."""
    dirs = fibonacci_sphere(n_dirs)
    roll_step = 360.0 / n_rolls
    mats, dir_idx, roll_idx, rolls_deg = [], [], [], []
    for i, v in enumerate(dirs):
        R_align = rot_align_z_to(v)
        offset = (i * GOLDEN % 1.0) * roll_step
        for j in range(n_rolls):
            theta = j * roll_step + offset
            mats.append(R_align @ Rotation.from_euler('z', theta, degrees=True).as_matrix())
            dir_idx.append(i); roll_idx.append(j); rolls_deg.append(theta)
    mats = np.array(mats)
    return dict(
        rotations=mats, quats=Rotation.from_matrix(mats).as_quat(), dirs=dirs,
        dir_idx=np.array(dir_idx), roll_idx=np.array(roll_idx),
        rolls_deg=np.array(rolls_deg), n_dirs=n_dirs, n_rolls=n_rolls,
    )

def nearest_rotation_idx(R_queries, quats_set):
    """Geodesic NN of each query rotation in the set → (indices, dist_deg)."""
    q = Rotation.from_matrix(R_queries).as_quat()        # (M, 4)
    dots = np.abs(q @ quats_set.T)                       # (M, N)
    idx = dots.argmax(axis=1)
    ang = 2 * np.degrees(np.arccos(np.clip(dots[np.arange(len(idx)), idx], -1.0, 1.0)))
    return idx, ang

# ── Build + sanity checks ─────────────────────────────────────────────────────
N_DIRS, N_ROLLS = 64, 8        # 512 distinct rotations — same budget as the 45° Euler grid
hopf = build_hopf_rotation_set(N_DIRS, N_ROLLS)
RotN = len(hopf['rotations'])

pair_dots = np.abs(hopf['quats'] @ hopf['quats'].T)
np.fill_diagonal(pair_dots, 0.0)
min_sep = 2 * np.degrees(np.arccos(np.clip(pair_dots.max(), -1.0, 1.0)))
dets = np.linalg.det(hopf['rotations'])
print(f'{RotN} rotations   det ∈ [{dets.min():.6f}, {dets.max():.6f}]')
print(f'Min pairwise geodesic separation: {min_sep:.1f}°   (Euler grid had exact duplicates → 0°)')

probe = Rotation.random(20000, random_state=0).as_matrix()
_, cover_d = nearest_rotation_idx(probe, hopf['quats'])
print(f'Covering radius (random rotation → nearest test rotation): '
      f'mean={cover_d.mean():.1f}°  p95={np.percentile(cover_d, 95):.1f}°  max={cover_d.max():.1f}°')


In [ ]:

# ── 8b: Generic sweep over an explicit rotation set ───────────────────────────
def run_rotation_sweep(src_base_np_in, rot_set, ref_pts=None,
                       gt_rotation=None, cache_path=None, desc='sweep'):
    """Like run_euler_sweep, but over rot_set['rotations'] instead of an Euler
    grid.  Residual semantics identical: perfect prediction → 0° regardless of
    the gt_rotation baked into src."""
    if cache_path and os.path.exists(cache_path):
        print(f'Loading cached sweep from {cache_path}')
        with open(cache_path, 'rb') as f:
            return pickle.load(f)

    if ref_pts is None:
        ref_pts = mean_ref_pts

    mats = rot_set['rotations'].astype(np.float32)
    n = len(mats)
    residuals  = np.zeros(n, dtype=np.float32)
    nn_dists   = np.zeros(n, dtype=np.float32)
    raw_angles = np.zeros(n, dtype=np.float32)

    src_base_t = torch.from_numpy(src_base_np_in).float().to(DEVICE)

    for i in tqdm(range(n), desc=desc):
        R_sweep = mats[i]
        src_rot = src_base_t @ torch.from_numpy(R_sweep).to(DEVICE).T

        out = run_bypass(src_rot, ref_pts)
        T_pred = out['estimated_transform']
        R_pred = T_pred[:3, :3].cpu().numpy()

        R_effective = R_sweep @ gt_rotation.T if gt_rotation is not None else R_sweep
        residuals[i]  = residual_angle_deg(R_pred, R_effective)
        raw_angles[i] = rot_angle_deg(T_pred)

        src_aln = apply_transform(out['src_pts_0'], T_pred).cpu().numpy()
        nn_dists[i] = nn_dist(src_aln, out['ref_pts_0'].cpu().numpy())

    result = dict(
        rotations=rot_set['rotations'], quats=rot_set['quats'],
        dirs=rot_set['dirs'], dir_idx=rot_set['dir_idx'],
        roll_idx=rot_set['roll_idx'], rolls_deg=rot_set['rolls_deg'],
        n_dirs=rot_set['n_dirs'], n_rolls=rot_set['n_rolls'],
        residuals=residuals, nn_dists=nn_dists, raw_angles=raw_angles,
        gt_rotation=gt_rotation,
    )
    if cache_path:
        with open(cache_path, 'wb') as f:
            pickle.dump(result, f)
        print(f'Saved to {cache_path}')
    return result

print('run_rotation_sweep ready.')


In [ ]:

# ── 8c: Sphere-map plot helper ────────────────────────────────────────────────
def plot_sphere_maps(values, rot_set, title, cmap_use=None, norm_use=None,
                     thresh=None, cbar_label='Residual angle (deg)'):
    """One Mollweide panel per roll bin + one mean-over-rolls panel.
    Dot position = direction the face's +z axis points; color = `values` for
    that (direction, roll).  NaN values are drawn grey ("no data")."""
    n_dirs, n_rolls = rot_set['n_dirs'], rot_set['n_rolls']
    dirs = rot_set['dirs']
    lon = np.arctan2(dirs[:, 1], dirs[:, 0])
    lat = np.arcsin(np.clip(dirs[:, 2], -1.0, 1.0))

    vals_grid = np.full((n_dirs, n_rolls), np.nan)
    vals_grid[rot_set['dir_idx'], rot_set['roll_idx']] = values
    roll_step = 360.0 / n_rolls

    npanels = n_rolls + 1
    ncols = 3
    nrows = int(np.ceil(npanels / ncols))
    fig = plt.figure(figsize=(5.2 * ncols, 2.9 * nrows))
    fig.suptitle(title, fontsize=12)
    sc_last = None
    for p in range(npanels):
        ax = fig.add_subplot(nrows, ncols, p + 1, projection='mollweide')
        if p < n_rolls:
            v = vals_grid[:, p]
            lbl = f'roll bin {p}  (≈{p * roll_step:.0f}–{(p + 1) * roll_step:.0f}°)'
        else:
            with np.errstate(invalid='ignore'):
                v = np.nanmean(vals_grid, axis=1)
            lbl = 'mean over rolls'
        ok_mask = np.isfinite(v)
        if (~ok_mask).any():
            ax.scatter(lon[~ok_mask], lat[~ok_mask], c='lightgrey', s=24)
        if ok_mask.any():
            sc_last = ax.scatter(lon[ok_mask], lat[ok_mask], c=v[ok_mask],
                                 cmap=cmap_use, norm=norm_use, s=24)
            stats = f'mean={np.mean(v[ok_mask]):.0f}°'
            if thresh is not None:
                stats += f'  ok={100 * np.mean(v[ok_mask] < thresh):.0f}%'
        else:
            stats = 'no data'
        ax.set_title(f'{lbl}\n{stats}', fontsize=9)
        ax.grid(alpha=0.3)
        ax.set_xticklabels([]); ax.set_yticklabels([])
    fig.subplots_adjust(right=0.90, hspace=0.35)
    if sc_last is not None:
        cax = fig.add_axes([0.92, 0.15, 0.013, 0.70])
        cb = fig.colorbar(sc_last, cax=cax)
        cb.set_label(cbar_label, fontsize=10)
    plt.show()

print('plot_sphere_maps ready.')


In [ ]:

# ── 8d: Run Hopf sweeps — all real scans + val pairs ──────────────────────────
N_VAL_PAIRS_HOPF = 10
REAL_SCANS_DIR   = os.path.join(ROOT_DIR, 'planck_scans', 'npy_scaled_uniform')
val_dataset      = val_loader.dataset
hopf_tag         = f'hopf{N_DIRS}x{N_ROLLS}'

hopf_sweeps = {}

scan_files = sorted(f for f in os.listdir(REAL_SCANS_DIR) if f.endswith('.npy'))
print(f'Found {len(scan_files)} real scans in {REAL_SCANS_DIR}')
for fname in scan_files:
    pts = np.load(os.path.join(REAL_SCANS_DIR, fname))[:, :3].astype(np.float32)
    pts = pts - pts.mean(axis=0)
    key = os.path.splitext(fname)[0]
    cache = os.path.join(CACHE_DIR, f'{key}_{hopf_tag}.pkl')
    hopf_sweeps[key] = run_rotation_sweep(pts, hopf, cache_path=cache, desc=key)

for i in range(N_VAL_PAIRS_HOPF):
    raw_i  = val_dataset[i]
    src_i  = raw_i['src_points'].astype(np.float32)
    ref_i  = torch.from_numpy(raw_i['ref_points'].astype(np.float32)).float().to(DEVICE)
    R_gt_i = raw_i['transform'][:3, :3].astype(np.float32)
    key = f'val_{i:02d}'
    cache = os.path.join(CACHE_DIR, f'{key}_{hopf_tag}.pkl')
    hopf_sweeps[key] = run_rotation_sweep(
        src_i - src_i.mean(axis=0), hopf, ref_pts=ref_i,
        gt_rotation=R_gt_i, cache_path=cache, desc=key)

print(f'\n{len(hopf_sweeps)} Hopf sweeps ready.')


In [ ]:

# ── 8e: Summary + scan-vs-val comparison via quaternion NN re-indexing ────────
THRESH_H = 30

print(f'{"Sample":25s}  {"Mean res":>9s}  {"Median":>8s}  {"<30° %":>7s}  {"<15° %":>7s}  {"Mean NN":>9s}')
print('-' * 78)
for key, sw in hopf_sweeps.items():
    r = sw['residuals']
    print(f'{key:25s}  {r.mean():8.1f}°  {np.median(r):7.1f}°  '
          f'{100 * (r < 30).mean():6.1f}%  {100 * (r < 15).mean():6.1f}%  {sw["nn_dists"].mean():8.4f} m')

real_keys_h = [k for k in hopf_sweeps if not k.startswith('val_')]
val_keys_h  = [k for k in hopf_sweeps if k.startswith('val_')]

# Real scans: the tested orientation IS the set rotation → direct average.
scan_mean_h = np.stack([hopf_sweeps[k]['residuals'] for k in real_keys_h]).mean(axis=0)

# Val pairs: src carries R_gt (transform[:3,:3], ref ≈ R_gt @ src), so after
# pre-rotation the actual face orientation is R_sweep @ R_gt.T — note the
# transpose: R_gt maps src→ref, orientation is the ref→src direction.
# Re-index each cell to the geodesically nearest rotation in the test set.
val_accum = np.zeros(RotN)
val_count = np.zeros(RotN, dtype=int)
for k in val_keys_h:
    sw = hopf_sweeps[k]
    R_eff = np.einsum('nij,kj->nik', sw['rotations'], sw['gt_rotation'].astype(np.float64))
    idx, snap_d = nearest_rotation_idx(R_eff, hopf['quats'])
    np.add.at(val_accum, idx, sw['residuals'].astype(np.float64))
    np.add.at(val_count, idx, 1)
    print(f'{k}: mean snap offset {snap_d.mean():.1f}°  '
          f'(cumulative coverage {100 * (val_count > 0).mean():.1f}%)')

val_mean_h = np.where(val_count > 0, val_accum / np.maximum(val_count, 1), np.nan)
print(f'\nVal coverage: {100 * np.isfinite(val_mean_h).mean():.1f}% of test rotations  '
      f'(avg {val_count[val_count > 0].mean():.1f} samples/rotation)')
print(f'Scan mean residual: {scan_mean_h.mean():.1f}°   '
      f'Val mean residual: {np.nanmean(val_mean_h):.1f}°')

cmap_h = plt.cm.RdYlGn_r
norm_h = mcolors.Normalize(vmin=0, vmax=180)

plot_sphere_maps(scan_mean_h, hopf,
                 f'Real scans ({len(real_keys_h)}) — mean residual per orientation  '
                 f'({N_DIRS} dirs × {N_ROLLS} rolls)',
                 cmap_use=cmap_h, norm_use=norm_h, thresh=THRESH_H)

plot_sphere_maps(val_mean_h, hopf,
                 f'Validation ({len(val_keys_h)} pairs, geodesic-NN re-indexed) — mean residual\n'
                 f'grey = orientation not hit by any val pair',
                 cmap_use=cmap_h, norm_use=norm_h, thresh=THRESH_H)

diff_h = scan_mean_h - val_mean_h
plot_sphere_maps(diff_h, hopf,
                 'Difference: scans − val   (red = scans worse → domain gap, blue = val worse)',
                 cmap_use='RdBu_r',
                 norm_use=mcolors.TwoSlopeNorm(vmin=-90, vcenter=0, vmax=90),
                 cbar_label='Δ residual (deg)')
